# S&P 500 multivariate backtest demo

**Workflow:** edit the next code cell (`DEMO CONFIG`) to choose which models run and set the backtest window, then **Run All**.

One table at the end lists every run: CRPS, counts, and whether covariates were used.

**Leakage safeguards** (in `stock_price_forecasting_multivariate/data.py`):

- covariates are shifted one business day and use conservative macro release proxies where applicable
- evaluation uses `released_at <= as_of` via `ForecastContext`


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def _bootstrap_sys_path() -> Path:
    if os.environ.get("AIENG_REPO_ROOT"):
        root = Path(os.environ["AIENG_REPO_ROOT"]).expanduser().resolve()
        if (root / "aieng-forecasting").is_dir() and (root / "implementations").is_dir():
            for sub in (root, root / "aieng-forecasting", root / "implementations"):
                s = str(sub)
                if s not in sys.path:
                    sys.path.insert(0, s)
            return root

    seeds: list[Path] = []
    try:
        from IPython import get_ipython

        ip = get_ipython()
        if ip is not None:
            nb = ip.user_ns.get("__vsc_ipynb_file__")
            if nb:
                seeds.append(Path(nb).resolve().parent)
    except Exception:
        pass

    for env_key in ("PWD", "INIT_CWD", "OLDPWD"):
        v = os.environ.get(env_key)
        if v:
            try:
                seeds.append(Path(v).expanduser().resolve())
            except (OSError, ValueError):
                pass

    seeds.append(Path.cwd().resolve())

    for seed in seeds:
        for p in (seed, *seed.parents):
            if (p / "aieng-forecasting").is_dir() and (p / "implementations").is_dir():
                for sub in (p, p / "aieng-forecasting", p / "implementations"):
                    s = str(sub)
                    if s not in sys.path:
                        sys.path.insert(0, s)
                return p

    raise RuntimeError("Cannot find repo root. Pick the project's .venv kernel, or set AIENG_REPO_ROOT.")


REPO_ROOT = _bootstrap_sys_path()
REPO_ROOT


In [ ]:
# =============================================================================
# DEMO CONFIG — edit flags and windows here only
# =============================================================================

# Which rows appear in the final leaderboard (True = run backtest for that row)
RUN_MODELS: dict[str, bool] = {
    "linreg_target_only": True,
    "linreg_with_covariates": True,
    "lightgbm_target_only": False,  # can hard-crash the kernel on macOS (native segfault)
    "lightgbm_with_covariates": False,
}

# Backtest window (business-day origins between start and end, inclusive)
SPEC_START = "2026-04-18"
SPEC_END = "2026-04-30"
SPEC_STRIDE = 1
SPEC_WARMUP = 3

# Data pulled into each service (wide history helps warmup inside the window)
DATA_HISTORY_START = "2000-01-01"
REFRESH_CACHE = False

# Predictor speed / quality knobs
QUICK_LAGS = 5
QUICK_LAGS_PAST_COV = 5
QUICK_NUM_SAMPLES = 50

# Subset of covariates for the "with_covariates" runs (must exist in svc_cov after build)
FAST_COVARIATE_SERIES_IDS = [
    "vix_level_l1b",
    "vix_log_ret_1b_l1b",
    "nasdaq_log_ret_1b_l1b",
    "ust2y10y_spread_l1b",
]

SHOW_BACKTEST_PROGRESS = True

# LightGBM: passed through to Darts (single-thread lowers some crash modes; not a guarantee)
LIGHTGBM_KWARGS = {"num_threads": 1, "verbosity": -1}


In [ ]:
import pandas as pd

from aieng.forecasting.evaluation import BacktestSpec, ForecastingTask, backtest
from implementations.experiments.stock_price_forecasting_multivariate import (
    DEFAULT_COVARIATE_SERIES_IDS,
    SP500_LOG_RETURN_SERIES_ID,
    build_sp500_multivariate_service,
)
from methods.darts_regression import DartsLightGBMPredictor, DartsLinearRegressionPredictor


## Build data services


In [ ]:
svc_no_cov = build_sp500_multivariate_service(
    include_covariates=False,
    start=DATA_HISTORY_START,
    end=None,
    refresh=REFRESH_CACHE,
)

svc_cov = build_sp500_multivariate_service(
    include_covariates=True,
    covariate_series_ids=DEFAULT_COVARIATE_SERIES_IDS,
    start=DATA_HISTORY_START,
    end=None,
    refresh=REFRESH_CACHE,
)

registered_covariates = [
    sid for sid in DEFAULT_COVARIATE_SERIES_IDS if sid in set(svc_cov.series_ids)
]
selected_covariates = [sid for sid in FAST_COVARIATE_SERIES_IDS if sid in registered_covariates]
if not selected_covariates:
    selected_covariates = registered_covariates

print("Target-only service:", svc_no_cov.series_ids)
print("Covariate service (count):", len(svc_cov.series_ids), svc_cov.series_ids)
print("Selected covariates for multivariate runs:", selected_covariates)


## Task and backtest spec


In [ ]:
task = ForecastingTask(
    task_id="sp500_log_return_1b",
    target_series_id=SP500_LOG_RETURN_SERIES_ID,
    horizons=[1],
    frequency="B",
    description="Forecast next-session open vs prior close S&P500 log return.",
)

spec = BacktestSpec(
    task=task,
    start=SPEC_START,
    end=SPEC_END,
    stride=SPEC_STRIDE,
    warmup=SPEC_WARMUP,
    description="Multivariate demo: models selected in DEMO CONFIG.",
)

spec


## Run selected models → unified results

Each enabled entry in `RUN_MODELS` runs one `backtest` and appends a row to `RESULTS_DF` (sorted by mean CRPS).


In [ ]:
def _run_one_row(*, run_key: str, predictor, svc) -> dict[str, object]:
    result = backtest(
        predictor=predictor,
        spec=spec,
        data_service=svc,
        show_progress=SHOW_BACKTEST_PROGRESS,
    )
    uses_cov = svc is svc_cov and run_key != "linreg_target_only" and run_key != "lightgbm_target_only"
    cov_ids = selected_covariates if uses_cov else []
    return {
        "run_key": run_key,
        "model": run_key.replace("_", " "),
        "uses_covariates": bool(uses_cov),
        "n_covariates": len(cov_ids) if uses_cov else 0,
        "covariates": ", ".join(cov_ids) if cov_ids else "—",
        "predictor_id": result.predictor_id,
        "mean_crps": float(result.mean_crps),
        "n_scores": int(len(result.scores)),
        "n_predictions": int(len(result.predictions)),
        "skipped_origins": int(result.skipped_origins),
    }


def _predictor_for(run_key: str):
    if run_key == "linreg_target_only":
        return DartsLinearRegressionPredictor(
            lags=QUICK_LAGS,
            covariate_series_ids=None,
            num_samples=QUICK_NUM_SAMPLES,
        )
    if run_key == "linreg_with_covariates":
        return DartsLinearRegressionPredictor(
            lags=QUICK_LAGS,
            lags_past_covariates=QUICK_LAGS_PAST_COV,
            covariate_series_ids=selected_covariates,
            num_samples=QUICK_NUM_SAMPLES,
        )
    if run_key == "lightgbm_target_only":
        return DartsLightGBMPredictor(
            lags=QUICK_LAGS,
            covariate_series_ids=None,
            num_samples=QUICK_NUM_SAMPLES,
            lgbm_kwargs=dict(LIGHTGBM_KWARGS),
        )
    if run_key == "lightgbm_with_covariates":
        return DartsLightGBMPredictor(
            lags=QUICK_LAGS,
            lags_past_covariates=QUICK_LAGS_PAST_COV,
            covariate_series_ids=selected_covariates,
            num_samples=QUICK_NUM_SAMPLES,
            lgbm_kwargs=dict(LIGHTGBM_KWARGS),
        )
    raise KeyError(f"Unknown run_key: {run_key!r}")


def _service_for(run_key: str):
    if run_key in ("linreg_target_only", "lightgbm_target_only"):
        return svc_no_cov
    if run_key in ("linreg_with_covariates", "lightgbm_with_covariates"):
        return svc_cov
    raise KeyError(run_key)


rows: list[dict[str, object]] = []
for run_key, enabled in RUN_MODELS.items():
    if not enabled:
        continue
    try:
        pred = _predictor_for(run_key)
        svc = _service_for(run_key)
        rows.append(_run_one_row(run_key=run_key, predictor=pred, svc=svc))
    except Exception as exc:
        rows.append(
            {
                "run_key": run_key,
                "model": run_key.replace("_", " "),
                "uses_covariates": run_key.endswith("with_covariates"),
                "n_covariates": len(selected_covariates) if run_key.endswith("with_covariates") else 0,
                "covariates": "—",
                "predictor_id": "error",
                "mean_crps": float("nan"),
                "n_scores": 0,
                "n_predictions": 0,
                "skipped_origins": 0,
                "error": str(exc),
            }
        )

RESULTS_DF = pd.DataFrame(rows).sort_values("mean_crps", na_position="last")
RESULTS_DF


### Notes

- **LightGBM** may **kill the kernel** on some macOS setups; keep those flags `False` unless you accept that risk (`brew install libomp` sometimes helps).
- Increase `QUICK_LAGS`, `QUICK_NUM_SAMPLES`, and widen `SPEC_*` for more serious runs (slower).
